# Roomify — Colab Launcher

Run cells **top to bottom** on a fresh session.  After Cell 6 prints a `trycloudflare.com` URL, open it in your laptop browser.

| Cell | Purpose | One-time? |
|------|---------|----------|
| 1 | Mount Google Drive, create folder layout | No (fast) |
| 2 | Clone repo + install deps | First run per Drive |
| 2b | Unzip SUN RGB-D + build subset + manifest | **Yes — first run only** |
| 3 | Set HF_HOME to Drive-backed cache | No (fast) |
| 4 | Verify GPU | No (fast) |
| 5 | SD 1.5 smoke-test generation | First run (downloads weights) |
| 6 | Start Streamlit + Cloudflare tunnel | Every session |
| 7 | Reconnect helper (after disconnect) | On reconnect only |

In [ ]:
# ── Cell 1: Mount Google Drive and create folder structure ─────────────────
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/roomify')
for subdir in ['data', 'outputs', 'hf_cache']:
    (DRIVE_ROOT / subdir).mkdir(parents=True, exist_ok=True)

print('Drive mounted.  Folder layout:')
for p in sorted(DRIVE_ROOT.iterdir()):
    print(' ', p)

In [ ]:
# ── Cell 2: Clone repo and install Python dependencies ─────────────────────
from pathlib import Path

REPO_URL = 'https://github.com/ben-blake/roomify.git'
REPO_DIR = Path('/content/roomify')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'{REPO_DIR} already exists — pulling latest changes.')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

# Symlink Drive data/outputs into the repo so paths resolve correctly.
DRIVE_ROOT = Path('/content/drive/MyDrive/roomify')
for link_name in ['data', 'outputs']:
    link_path = REPO_DIR / link_name
    target = DRIVE_ROOT / link_name
    if not link_path.exists():
        link_path.symlink_to(target)
        print(f'Symlinked {link_path} -> {target}')

!pip install -q -r requirements.txt
print('\nInstallation complete.')

In [ ]:
# ── Cell 2b: One-time SUN RGB-D setup (skip if manifest.csv already exists) ─
#
# Before running this cell:
#   1. Download SUNRGBD V1 from https://rgbd.cs.princeton.edu/
#   2. Upload the zip to Google Drive at:
#      MyDrive/roomify/data/SUNRGBD.zip
#
# This cell is safe to re-run — it skips any step that is already done.

import zipfile, subprocess, sys
from pathlib import Path

DRIVE_DATA = Path('/content/drive/MyDrive/roomify/data')
ZIP_PATH   = DRIVE_DATA / 'SUNRGBD.zip'
RAW_DIR    = DRIVE_DATA / 'SUNRGBD'
SUBSET_DIR = DRIVE_DATA / 'sunrgbd_subset'
MANIFEST   = SUBSET_DIR / 'manifest.csv'
REPO_DIR   = Path('/content/roomify')

# ── Step 1: Unzip ──────────────────────────────────────────────────────────
if RAW_DIR.exists():
    print(f'Raw dataset already present at {RAW_DIR} — skipping unzip.')
elif not ZIP_PATH.exists():
    print(f'ERROR: {ZIP_PATH} not found.')
    print('Upload SUNRGBD.zip to MyDrive/roomify/data/ first, then re-run this cell.')
else:
    print(f'Unzipping {ZIP_PATH} -> {RAW_DIR} ...')
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(RAW_DIR)
    print('Unzip complete.')

# ── Step 2: Build subset + manifest ────────────────────────────────────────
if MANIFEST.exists():
    import pandas as pd
    n = len(pd.read_csv(MANIFEST))
    print(f'Manifest already exists ({n} records) — skipping buildSubset.')
elif not RAW_DIR.exists():
    print('Skipping buildSubset — raw dataset not ready yet.')
else:
    print('Building subset (~200 samples across 5 scene types)...')
    result = subprocess.run(
        [
            sys.executable,
            str(REPO_DIR / 'scripts' / 'buildSubset.py'),
            '--sunrgbd-root', str(RAW_DIR),
            '--output-dir',   str(SUBSET_DIR),
            '--samples-per-scene', '40',
            '--copy',
        ],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('ERROR:', result.stderr)
    else:
        print('Subset built successfully.')

print('\nData setup status:')
print(f'  Raw dataset : {"present" if RAW_DIR.exists() else "MISSING"}')
print(f'  Manifest    : {"present" if MANIFEST.exists() else "MISSING"}')

In [ ]:
# ── Cell 3: Point HF_HOME at the Drive-backed cache ────────────────────────
import os
from pathlib import Path

HF_CACHE = Path('/content/drive/MyDrive/roomify/hf_cache')
HF_CACHE.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE / 'transformers')

print(f'HF_HOME={os.environ["HF_HOME"]}')
print(f'Cache size: ', end='')
!du -sh {HF_CACHE} 2>/dev/null || echo '(empty)'

In [ ]:
# ── Cell 4: Verify GPU and log to session-level file ───────────────────────
import subprocess, json, datetime
from pathlib import Path

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
gpu_info = result.stdout.strip() if result.returncode == 0 else 'GPU not found'
print('GPU:', gpu_info)

log = {
    'timestamp': datetime.datetime.utcnow().isoformat(),
    'gpu': gpu_info,
}
log_path = Path('/content/roomify/session_gpu.json')
log_path.write_text(json.dumps(log, indent=2))
print('Logged to', log_path)

In [ ]:
# ── Cell 5: SD 1.5 smoke-test (downloads weights on first run) ─────────────
import sys, os
sys.path.insert(0, '/content/roomify/src')

import torch
from diffusers import StableDiffusionPipeline
from pathlib import Path
from IPython.display import display

pipe = StableDiffusionPipeline.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5',
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe = pipe.to('cuda')
pipe.enable_attention_slicing()

generator = torch.Generator(device='cuda').manual_seed(42)
image = pipe(
    'A scandinavian bedroom, natural light, 4k interior design photography',
    negative_prompt='low quality, blurry, distorted proportions',
    num_inference_steps=20,
    generator=generator,
).images[0]

out_path = Path('/content/roomify/outputs/smoke_test.png')
out_path.parent.mkdir(parents=True, exist_ok=True)
image.save(out_path)
print('Smoke test image saved to', out_path)
display(image)

In [ ]:
# ── Cell 6: Start Streamlit + Cloudflare quick tunnel ──────────────────────
import subprocess, time, re
from pathlib import Path

REPO_DIR = Path('/content/roomify')
PORT = 8501

# -- Install cloudflared binary if missing --
CF_BIN = Path('/usr/local/bin/cloudflared')
if not CF_BIN.exists():
    print('Downloading cloudflared...')
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {CF_BIN}
    !chmod +x {CF_BIN}
    print('cloudflared installed.')

# -- Launch Streamlit in background --
st_log = open('/content/streamlit.log', 'w')
st_proc = subprocess.Popen(
    ['python', '-m', 'streamlit', 'run', str(REPO_DIR / 'app.py'),
     '--server.port', str(PORT), '--server.headless', 'true'],
    cwd=str(REPO_DIR),
    stdout=st_log, stderr=st_log,
)
print(f'Streamlit PID {st_proc.pid} starting on port {PORT}...')
time.sleep(4)

# -- Launch Cloudflare tunnel --
cf_proc = subprocess.Popen(
    [str(CF_BIN), 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=open('/content/cloudflared.log', 'w'), stderr=subprocess.PIPE,
    text=True,
)

tunnel_url = None
for _ in range(30):
    line = cf_proc.stderr.readline()
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print(f'\n=== Roomify is live at: {tunnel_url} ===')
else:
    print('WARNING: Could not parse tunnel URL. Check /content/cloudflared.log')

# -- pyngrok fallback (commented out; enable if cloudflared fails) --
# from pyngrok import ngrok
# ngrok.set_auth_token('YOUR_NGROK_AUTH_TOKEN')
# tunnel = ngrok.connect(PORT)
# print(f'ngrok URL: {tunnel.public_url}')

In [ ]:
# ── Cell 7: Reconnect helper ────────────────────────────────────────────────
# Run ONLY after a Colab disconnect — not on a fresh session.
import os, subprocess, time, re
from pathlib import Path
from google.colab import drive

print('Re-mounting Google Drive...')
drive.mount('/content/drive', force_remount=True)

os.environ['HF_HOME'] = '/content/drive/MyDrive/roomify/hf_cache'

!pkill -f streamlit 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
time.sleep(2)

REPO_DIR = Path('/content/roomify')
PORT = 8501
CF_BIN = Path('/usr/local/bin/cloudflared')

%cd {REPO_DIR}

st_proc = subprocess.Popen(
    ['python', '-m', 'streamlit', 'run', str(REPO_DIR / 'app.py'),
     '--server.port', str(PORT), '--server.headless', 'true'],
    cwd=str(REPO_DIR),
    stdout=open('/content/streamlit.log', 'w'),
    stderr=subprocess.STDOUT,
)
time.sleep(4)

cf_proc = subprocess.Popen(
    [str(CF_BIN), 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
)
tunnel_url = None
for _ in range(30):
    line = cf_proc.stderr.readline()
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print(f'\n=== Roomify reconnected at: {tunnel_url} ===')
else:
    print('WARNING: Could not parse tunnel URL. Check /content/cloudflared.log')